# Compare Summarizers

This notebook evaluates zero shot and few shot summarization using OpenRouter alongside our fine tuned BART and FLAN-T5 models pulled from Hugging Face. If `OR_KEY` is left blank, OpenRouter inference is skipped and only BART and FLAN run.

The notebook uses DeepSeek by default to minimize API costs. At current pricing, a full run with our ruleset and prompts costs approximately $0.004. Larger models increase this significantly; testing showed that Claude Sonnet and Opus produce the strongest results, but at a cost that is not sustainable for casual use. Avoid them unless you have a specific reason. THEY WILL BANKRUPT YOU.

Input is drawn as a random contiguous slice from `test.txt`, a file containing approximately 70,000 chat messages scraped from a Northernlion VOD. The window size is controlled by `SAMPLE_MSGS`. We recommend keeping it between 150 and 400 messages. Windows below 100 noticeably degraded BART output quality, though FLAN-T5 was less sensitive to window size.

In [ ]:
from pathlib import Path

IN_TXT = Path("test.txt")
OUT_JSON = Path("test_out.json")

# leave blank to skip OpenRouter
OR_KEY = ""

ZERO_MDL = "deepseek/deepseek-v3.2"
FEW_MDL = "deepseek/deepseek-chat"
REF = "http://localhost"
TITLE = "4-way Twitch Summary Compare"
OR_MAX_TOK = 96
OR_TEMP = 0.2

RULES_PATH = Path("ABSTRACTIVE_SUMMARY_RULESET.md")
PAIR_PATH = Path("summary_pairs.jsonl")
FEW_K = 3
POOL_LIM = 0
EX_LINES = 30
PROMPT_LINES = 220

BART_ID = "JDMates/TwitchBART"
FLAN_ID = "JDMates/TwitchFlanSmall-v2"

MAX_SRC = 1024
MAX_TGT = 64
BEAMS = 4
NO_REP_N = 3
LEN_PEN = 1.0
MAX_OUT_SENTS = 2
SHOW_CHUNKS = False

CLEAN = True
STRIP_META = True
DROP_SYS = True
COLLAPSE_DUPS = True
DUP_TH = 3
COLLAPSE_FLOODS = True
FLOOD_TH = 4
MIN_MSG_CHARS = 2
MAX_MSGS = 0

# contiguous window settings
SAMPLE_MSGS = 200
SAMPLE_SEED = None


In [7]:
import json
import math
import random
import re
import time
import urllib.error
import urllib.request
from collections import Counter
from dataclasses import dataclass
from typing import Any

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from chat_preprocess import DistillOpts, distill_text
from summarize_chat import clean_sum, hier_sum, split_chunks

OR_URL = "https://openrouter.ai/api/v1/chat/completions"

STOP = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "has", "he",
    "in", "is", "it", "its", "of", "on", "or", "that", "the", "to", "was", "were", "will",
    "with", "you", "your", "we", "they", "this", "those", "these", "i", "me", "my", "our",
}


@dataclass
class ShotEx:
    chat: str
    summary: str
    source_line: int


def read_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")
    text = path.read_text(encoding="utf-8", errors="replace").strip()
    if not text:
        raise ValueError(f"Input file is empty: {path}")
    return text


def tokenize(text: str) -> list[str]:
    toks = re.findall(r"[a-zA-Z0-9']+", text.lower())
    return [t for t in toks if len(t) > 1 and t not in STOP]


def tfidf_vectors(docs: list[list[str]]) -> tuple[list[dict[str, float]], dict[str, float]]:
    n_docs = len(docs)
    df = Counter()
    for d in docs:
        for tok in set(d):
            df[tok] += 1

    idf: dict[str, float] = {}
    for tok, c in df.items():
        idf[tok] = math.log((1 + n_docs) / (1 + c)) + 1.0

    vectors: list[dict[str, float]] = []
    for d in docs:
        tf = Counter(d)
        vec: dict[str, float] = {}
        norm = 0.0
        for tok, c in tf.items():
            v = (1 + math.log(c)) * idf.get(tok, 1.0)
            vec[tok] = v
            norm += v * v
        norm = math.sqrt(norm) if norm > 0 else 1.0
        for tok in list(vec.keys()):
            vec[tok] /= norm
        vectors.append(vec)
    return vectors, idf


def build_query_vector(tokens: list[str], idf: dict[str, float]) -> dict[str, float]:
    tf = Counter(tokens)
    vec: dict[str, float] = {}
    norm = 0.0
    for tok, c in tf.items():
        v = (1 + math.log(c)) * idf.get(tok, 1.0)
        vec[tok] = v
        norm += v * v
    norm = math.sqrt(norm) if norm > 0 else 1.0
    for tok in list(vec.keys()):
        vec[tok] /= norm
    return vec


def cosine_sparse(a: dict[str, float], b: dict[str, float]) -> float:
    if len(a) > len(b):
        a, b = b, a
    return sum(v * b.get(tok, 0.0) for tok, v in a.items())


def clean_out(text: str, max_sentences: int = 2) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return ""
    parts = re.split(r"(?<=[.!?])\s+", text)
    keep = [p.strip() for p in parts if p.strip()]
    if max_sentences > 0:
        keep = keep[:max_sentences]
    fixed: list[str] = []
    for s in keep:
        if s and s[-1] not in ".!?":
            s += "."
        fixed.append(s)
    return " ".join(fixed).strip()


def format_chat_for_prompt(chat_text: str, max_lines: int) -> str:
    lines = [ln.strip() for ln in chat_text.splitlines() if ln.strip()]
    if max_lines > 0:
        lines = lines[:max_lines]
    if not lines:
        return "- (no usable chat lines)"
    return "\n".join(f"- {ln}" for ln in lines)


def build_zero_prompt(distilled_chat: str, max_lines: int) -> str:
    lines_block = format_chat_for_prompt(distilled_chat, max_lines=max_lines)
    return "\n".join([
        "Task: Summarize this Twitch chat window.",
        "Output: 1-2 sentences, 20-55 words, plain text.",
        "Constraints: no usernames, no timestamps, no outside context, no bullet points, grounded in visible messages.",
        "",
        "Chat window:",
        lines_block,
        "",
        "Summary:",
    ])


def load_shots(path: Path, pool_limit: int) -> list[ShotEx]:
    out: list[ShotEx] = []
    with path.open("r", encoding="utf-8", errors="replace") as fh:
        for line_no, raw in enumerate(fh, start=1):
            line = raw.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            chat = str(obj.get("chat") or "").strip()
            summary = str(obj.get("summary") or "").strip()
            if chat and summary:
                out.append(ShotEx(chat=chat, summary=summary, source_line=line_no))
                if pool_limit > 0 and len(out) >= pool_limit:
                    break
    if not out:
        raise RuntimeError(f"No valid {{chat, summary}} rows found in {path}")
    return out


def pick_shots(distilled_chat: str, examples: list[ShotEx], k: int):
    docs = [tokenize(ex.chat) for ex in examples]
    vecs, idf = tfidf_vectors(docs)
    query_vec = build_query_vector(tokenize(distilled_chat), idf)
    sims = [(idx, cosine_sparse(query_vec, vecs[idx])) for idx in range(len(examples))]
    sims.sort(key=lambda x: x[1], reverse=True)

    selected: list[ShotEx] = []
    selected_meta: list[dict[str, Any]] = []
    for idx, score in sims[: max(1, k)]:
        ex = examples[idx]
        selected.append(ex)
        selected_meta.append({
            "source_line": ex.source_line,
            "similarity": round(score, 4),
            "summary_preview": ex.summary[:120],
        })
    return selected, selected_meta


def build_few_prompt(distilled_chat: str, ruleset_text: str, examples: list[ShotEx]) -> str:
    parts: list[str] = []
    parts.append("You must follow this ruleset while summarizing.")
    parts.append("Ruleset:")
    parts.append(ruleset_text.strip())
    parts.append("")
    parts.append("Few-shot examples:")

    for idx, ex in enumerate(examples, start=1):
        ex_text, _ = distill_text(
            ex.chat,
            options=DistillOpts(
                strip_metadata=STRIP_META,
                drop_system_lines=DROP_SYS,
                collapse_duplicate_messages=COLLAPSE_DUPS,
                duplicate_count_threshold=DUP_TH,
                collapse_token_floods=COLLAPSE_FLOODS,
                token_flood_threshold=FLOOD_TH,
                min_message_chars=MIN_MSG_CHARS,
                max_messages=EX_LINES,
            ),
        )
        parts.append(f"Example {idx} input:")
        parts.append(format_chat_for_prompt(ex_text, max_lines=EX_LINES))
        parts.append(f"Example {idx} output:")
        parts.append(ex.summary)
        parts.append("")

    parts.append("Now summarize this target chat.")
    parts.append("Target input:")
    parts.append(format_chat_for_prompt(distilled_chat, max_lines=PROMPT_LINES))
    parts.append("")
    parts.append("Output requirements: 1-2 sentences, concise, grounded, natural phrasing.")
    parts.append("Target output:")
    return "\n".join(parts)


def call_or(api_key: str, model: str, sys_prompt: str, usr_prompt: str):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": usr_prompt},
        ],
        "max_tokens": OR_MAX_TOK,
        "temperature": OR_TEMP,
    }
    req = urllib.request.Request(
        OR_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": REF,
            "X-Title": TITLE,
        },
        method="POST",
    )

    start = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=90) as resp:
            body = resp.read().decode("utf-8")
            data = json.loads(body)
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", errors="replace") if hasattr(e, "read") else str(e)
        raise RuntimeError(f"OpenRouter HTTP error {e.code}: {detail}") from e
    except urllib.error.URLError as e:
        raise RuntimeError(f"OpenRouter connection error: {e}") from e

    latency_ms = (time.perf_counter() - start) * 1000.0
    try:
        content = str(data["choices"][0]["message"]["content"])
    except Exception as e:
        raise RuntimeError(f"Unexpected OpenRouter response: {data}") from e

    usage = data.get("usage", {}) if isinstance(data, dict) else {}
    return clean_out(content, max_sentences=2), usage, latency_ms


def ensure_hf_id(model_id: str) -> None:
    if Path(model_id).exists():
        raise ValueError(
            f"Local path provided ('{model_id}'). This notebook is HF-only; set a Hugging Face model id like 'org/model'."
        )


def load_tok(model_id: str):
    ensure_hf_id(model_id)
    try:
        return AutoTokenizer.from_pretrained(model_id, local_files_only=False, use_fast=True)
    except Exception as e:
        msg = str(e)
        if "has no attribute 'keys'" in msg or "extra_special_tokens" in msg:
            return AutoTokenizer.from_pretrained(model_id, local_files_only=False, use_fast=False)
        raise


def run_hf(model_id: str, distilled_chat: str, device: torch.device):
    tokenizer = load_tok(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id, local_files_only=False)
    model.to(device)
    model.eval()

    chunks = split_chunks(
        text=distilled_chat,
        tokenizer=tokenizer,
        max_source_length=MAX_SRC,
    )

    start = time.perf_counter()
    summary = hier_sum(
        chunks=chunks,
        model=model,
        tokenizer=tokenizer,
        device=device,
        max_source_length=MAX_SRC,
        max_target_length=MAX_TGT,
        num_beams=BEAMS,
        no_repeat_ngram_size=NO_REP_N,
        length_penalty=LEN_PEN,
        show_chunk_info=SHOW_CHUNKS,
    )
    summary = clean_sum(summary, MAX_OUT_SENTS)
    latency_ms = (time.perf_counter() - start) * 1000.0
    return summary, {"model_id": model_id, "chunks": len(chunks), "latency_ms": round(latency_ms, 1)}


In [9]:
full_txt = read_text(IN_TXT)
all_lines = [ln for ln in full_txt.splitlines() if ln.strip()]
raw_n = len(all_lines)

if SAMPLE_MSGS > 0 and raw_n > SAMPLE_MSGS:
    rng = random.Random(SAMPLE_SEED) if SAMPLE_SEED is not None else random.Random()
    win = SAMPLE_MSGS
    start_i = rng.randint(0, raw_n - win)
    end_i = start_i + win
    samp_lines = all_lines[start_i:end_i]
else:
    start_i = 0
    end_i = raw_n
    samp_lines = all_lines

if samp_lines != all_lines[start_i:end_i]:
    raise RuntimeError("Sampling drifted from contiguous-window slicing. Check sampling logic.")

samp_n = len(samp_lines)
in_txt = "\n".join(samp_lines)

if CLEAN:
    chat_txt, clean_stats = distill_text(
        in_txt,
        options=DistillOpts(
            strip_metadata=STRIP_META,
            drop_system_lines=DROP_SYS,
            collapse_duplicate_messages=COLLAPSE_DUPS,
            duplicate_count_threshold=DUP_TH,
            collapse_token_floods=COLLAPSE_FLOODS,
            token_flood_threshold=FLOOD_TH,
            min_message_chars=MIN_MSG_CHARS,
            max_messages=MAX_MSGS,
        ),
    )
    chat_txt = chat_txt.strip()
    if not chat_txt:
        raise ValueError("Input empty after cleanup; loosen cleanup flags.")
else:
    chat_txt = in_txt
    clean_stats = None

zero_sum = None
few_sum = None
zero_m: dict[str, Any] = {}
few_m: dict[str, Any] = {}
zero_err = None
few_err = None

if OR_KEY.strip():
    try:
        zero_prompt = build_zero_prompt(chat_txt, max_lines=PROMPT_LINES)
        zero_sum, zero_use, zero_ms = call_or(
            api_key=OR_KEY.strip(),
            model=ZERO_MDL,
            sys_prompt="You are a concise Twitch chat summarizer. Return only summary text.",
            usr_prompt=zero_prompt,
        )
        zero_m = {"mdl": ZERO_MDL, "use": zero_use, "ms": round(zero_ms, 1), "skip": False}
    except Exception as e:
        zero_err = str(e)

    try:
        rules_txt = read_text(RULES_PATH)
        pool = load_shots(PAIR_PATH, POOL_LIM)
        shots, shot_m = pick_shots(chat_txt, pool, FEW_K)
        few_prompt = build_few_prompt(chat_txt, rules_txt, shots)
        few_sum, few_use, few_ms = call_or(
            api_key=OR_KEY.strip(),
            model=FEW_MDL,
            sys_prompt="You are a Twitch chat summarizer. Follow the rules. Return only summary text.",
            usr_prompt=few_prompt,
        )
        few_m = {
            "mdl": FEW_MDL,
            "use": few_use,
            "ms": round(few_ms, 1),
            "shots": shot_m,
            "skip": False,
        }
    except Exception as e:
        few_err = str(e)
else:
    zero_err = "Skipped: OR_KEY is blank"
    few_err = "Skipped: OR_KEY is blank"
    zero_m = {"mdl": ZERO_MDL, "skip": True}
    few_m = {"mdl": FEW_MDL, "skip": True}
    print("Skipping OpenRouter: OR_KEY is blank.")

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bart_sum, bart_m = run_hf(BART_ID, chat_txt, dev)
flan_sum, flan_m = run_hf(FLAN_ID, chat_txt, dev)

res = {
    "in": str(IN_TXT),
    "out": str(OUT_JSON),
    "ts": int(time.time()),
    "dev": str(dev),
    "cfg": {
        "clean": CLEAN,
        "prompt_lines": PROMPT_LINES,
        "few_k": FEW_K,
        "pool_lim": POOL_LIM,
        "ex_lines": EX_LINES,
        "or_max_tok": OR_MAX_TOK,
        "or_temp": OR_TEMP,
        "max_src": MAX_SRC,
        "max_tgt": MAX_TGT,
        "beams": BEAMS,
        "no_rep_n": NO_REP_N,
        "len_pen": LEN_PEN,
        "max_out_sents": MAX_OUT_SENTS,
        "sample_msgs": SAMPLE_MSGS,
        "sample_seed": SAMPLE_SEED,
    },
    "clean_stats": None if clean_stats is None else {
        "in_lines": clean_stats.input_lines,
        "out_lines": clean_stats.output_lines,
        "rm_sys": clean_stats.removed_system_lines,
        "dup_lines": clean_stats.collapsed_duplicate_lines,
        "dup_groups": clean_stats.collapsed_duplicate_groups,
        "flood_runs": clean_stats.collapsed_token_flood_runs,
        "rm_noise": clean_stats.removed_noise_lines,
        "trunc_lines": clean_stats.truncated_lines,
    },
    "meta": {
        "zero": zero_m,
        "few": few_m,
        "bart": bart_m,
        "flan": flan_m,
        "errs": {"zero": zero_err, "few": few_err},
        "raw_n": raw_n,
        "samp_n": samp_n,
        "win_start": start_i,
        "win_end": end_i,
    },
}

OUT_JSON.write_text(json.dumps(res, indent=2, ensure_ascii=False), encoding="utf-8")

print("=" * 64)
print("Done")
print("=" * 64)
print(f"in:  {IN_TXT}")
print(f"out: {OUT_JSON}")
print(f"sampled: {samp_n}/{raw_n} (window {start_i}:{end_i})")
if clean_stats is not None:
    print(
        f"clean lines {clean_stats.input_lines}->{clean_stats.output_lines}, "
        f"rm_sys={clean_stats.removed_system_lines}, "
        f"dups={clean_stats.collapsed_duplicate_lines}, "
        f"floods={clean_stats.collapsed_token_flood_runs}"
    )

print("\nZero:")
print(zero_sum if zero_sum else f"[skip/err] {zero_err}")
print("\nFew:")
print(few_sum if few_sum else f"[skip/err] {few_err}")
print("\nBART:")
print(bart_sum)
print("\nFLAN:")
print(flan_sum)


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1421.93it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Done
in:  test.txt
out: test_out.json
sampled: 200/68049 (window 25593:25793)
clean lines 200->161, rm_sys=4, dups=35, floods=5

Zero:
The chat is reacting to a streamer's gameplay, expressing mixed feelings about him playing a specific game while showing affection and teasing him. Many viewers reference past games and joke about pressuring him to play certain titles.

Few:
[skip/err] OpenRouter HTTP error 402: {"error":{"message":"Prompt tokens limit exceeded: 5200 > 3343. To increase, visit https://openrouter.ai/settings/credits and add more credits","code":402,"metadata":{"provider_name":null}},"user_id":"user_2x4mfaY815F7pq4whVcczLXgzqe"}

BART:
Chat keeps circling around Chib like a running joke and mostly reacts with laughter instead of taking it seriously. ICANT one of the most insane chib chats ever.

FLAN:
Chat keeps repeating Chib like a running joke and mostly reacts with laughter instead of taking it seriously. The mood is mostly amused and a little mean.
